# Feature engineering

Lags every feature so that the row for year *t* carries indicator values the model could
actually have observed before *t*. Two versions are built: t-1 for the main specifications
and t-2 for the longer-lag robustness test.

The lag is not the same thing as the train/test split. The split decides which rows a model
learns from; the lag decides which year's values feed a single prediction.

**Input** `data/merged/parquet_saves/merged_base.parquet`, `merged_robust.parquet`
**Output** `data/engineered/parquet_saves/me_merged_{base,robust}[_lag2].parquet`

From the original `10_featureengineering`.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

MERGED_DIR = Path("../data/merged/parquet_saves")
ENGINEERED_DIR = Path("../data/engineered/parquet_saves")
ENGINEERED_DIR.mkdir(parents=True, exist_ok=True)

PRIORITY_COLS = ["bank_id", "country_iso", "year", "crisis"] #identifiers and target first, every feature after them

## 1. Load the merged panels

Held unlagged in memory so the checks further down can compare against them without
re-reading from disk.

In [ ]:
orig_base = pd.read_parquet(MERGED_DIR / "merged_base.parquet")
orig_robust = pd.read_parquet(MERGED_DIR / "merged_robust.parquet")

print("Base:  ", orig_base.shape)
print("Robust:", orig_robust.shape)

## 2. Lagging

In [ ]:
def order_and_sort(df):
    df = df.sort_values(["bank_id", "year"]).reset_index(drop=True) #the shift below walks down the rows, so the order has to be right before it runs
    remaining = [c for c in df.columns if c not in PRIORITY_COLS]
    return df[PRIORITY_COLS + remaining]


def lag_features(df, periods):
    """Shift every feature back `periods` years within each bank."""
    df = order_and_sort(df)
    feature_cols = [c for c in df.columns if c not in PRIORITY_COLS]

    df[feature_cols] = df.groupby("bank_id")[feature_cols].shift(periods) #groupby keeps the shift inside each bank, so one bank's last year never becomes the next bank's first

    df = df.dropna(subset=feature_cols, how="all").reset_index(drop=True) #the first `periods` years of every bank now have no features at all, since no t-1 (or t-2) exists for them
    return df

## 3. t-1 lag, the main specification

In [ ]:
merged_base = lag_features(orig_base, 1)
merged_robust = lag_features(orig_robust, 1)

print("Base:  ", orig_base.shape, "->", merged_base.shape)
print("Robust:", orig_robust.shape, "->", merged_robust.shape)

### Checks

In [ ]:
def check_lag(orig, lagged, periods, name, col="r_cti"):
    print(f"=== {name}, t-{periods} ===")

    # 1) Exactly `periods` rows per bank should disappear, except for banks observed
    #    fewer years than that, which lose only the years they have. clip handles those.
    expected = orig.groupby("bank_id").size().clip(upper=periods).sum()
    dropped = len(orig) - len(lagged)
    print(f"Dropped {dropped} rows, expected {expected}: "
          f"{'OK' if dropped == expected else 'MISMATCH'}")

    # 2) Side by side for one bank: the value at year t in the lagged frame should be
    #    the value at year t-`periods` in the original.
    bank = lagged["bank_id"].iloc[0]
    o = orig.loc[orig["bank_id"] == bank, ["year", col]].sort_values("year").reset_index(drop=True)
    l = lagged.loc[lagged["bank_id"] == bank, ["year", col]].sort_values("year").reset_index(drop=True)
    print(f"\nBank {bank}, {col}")
    print(o.head(6).merge(l.head(6), on="year", how="outer", suffixes=("_orig", "_lagged")).to_string(index=False))


check_lag(orig_base, merged_base, 1, "baseline")
check_lag(orig_robust, merged_robust, 1, "robustness")

### Save

In [ ]:
merged_base.to_parquet(ENGINEERED_DIR / "me_merged_base.parquet")
merged_robust.to_parquet(ENGINEERED_DIR / "me_merged_robust.parquet")
print("Saved t-1 panels.")

## 4. t-2 lag, robustness

Tests whether the indicators are still informative when observed further ahead of the
crisis. Rebuilt from the unlagged originals, since applying another shift to the t-1 panel
would compound the two.

In [ ]:
base_lag2 = lag_features(orig_base, 2) #built from the originals, not from merged_base, which is already lagged once
robust_lag2 = lag_features(orig_robust, 2)

print("Base:  ", orig_base.shape, "->", base_lag2.shape)
print("Robust:", orig_robust.shape, "->", robust_lag2.shape)

check_lag(orig_base, base_lag2, 2, "baseline")
check_lag(orig_robust, robust_lag2, 2, "robustness")

### Save

In [ ]:
base_lag2.to_parquet(ENGINEERED_DIR / "me_merged_base_lag2.parquet")
robust_lag2.to_parquet(ENGINEERED_DIR / "me_merged_robust_lag2.parquet")
print("Saved t-2 panels.")